###  Query Similarity Analysis
Find similar SPARQL queries by comparing their DBpedia result sets (Jaccard) and query texts (Cosine)

In [ ]:
# 1. Import required libraries
import json
import multiprocessing
from tqdm import tqdm
import ijson
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import torch

In [ ]:
# 2. Load queries with valid answers and preprocess results
def extract_result_strings(entry):
    values = set()
    for result in entry.get("results", []):
        for k, v in result.items():
            if isinstance(v, dict) and "value" in v:
                values.add(f"{k}:{v['value']}")
    return values

INPUT_FILE = "queries_with_answers.json"  
MAX_ENTRIES = 50000  

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

if len(data) > MAX_ENTRIES:
    data = data[:MAX_ENTRIES]
print(f"Processing {len(data)} entries (capped to {MAX_ENTRIES}).")

# preprocessed: list of (idx, sparql_query, result_set)
preprocessed = [(i, entry.get("sparql_query", "").strip(), extract_result_strings(entry))
                for i, entry in enumerate(data)]

In [ ]:
# 3. Compute Jaccard Similarities (use threads for notebook stability)
from multiprocessing.dummy import Pool as ThreadPool  # threads safer in Jupyter/macOS

def process_query(args, max_similars=20, jaccard_threshold=0.7):
    base_id, base_query, base_results = args
    local_similars = []
    # compare only with subsequent items to avoid duplicate pairs
    for sim_id, sim_query, sim_results in preprocessed[base_id + 1:]:
        intersection = base_results & sim_results
        union = base_results | sim_results
        jaccard_score = len(intersection) / len(union) if union else 0.0
        if jaccard_score >= jaccard_threshold:
            local_similars.append({
                "similar_query": sim_query,
                "jaccard_score": round(jaccard_score, 3),
                "similar_results": list(sim_results)
            })
            if len(local_similars) >= max_similars:
                break
    if local_similars:
        return {
            "base_query": base_query,
            "base_results": list(base_results),
            "similar_queries": local_similars
        }
    return None

In [ ]:
max_similars = 20
jaccard_threshold = 0.7
args_list = preprocessed  # already (idx, query, results)
thread_count = min(8, max(1, multiprocessing.cpu_count() // 2, len(args_list)))  # conservative

results = []
with ThreadPool(thread_count) as pool:
    for res in tqdm(pool.imap_unordered(lambda a: process_query(a, max_similars, jaccard_threshold), args_list),
                    total=len(args_list), desc="Computing Jaccard similarities"):
        if res:
            results.append(res)

OUT_JACCARD = "jaccard_similarities_optimized_0_6.json"
with open(OUT_JACCARD, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"JSON file written: {OUT_JACCARD}")

In [ ]:
# 4. Compute Cosine Similarities with Embeddings (single clean pass)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)  # ensure model string is available

file_path = "jaccard_similarities_optimized_0_6.json"
output_file = "cosine_similarities_sorted_0_6.json"
threshold_cosine = 0.85  # set to None to disable cosine filtering

results_cosine = []
with open(file_path, "r", encoding="utf-8") as f:
    entries = json.load(f)

for obj in tqdm(entries, desc="Computing cosine similarities"):
    base_query = obj["base_query"]
    sim_objs = obj.get("similar_queries", [])
    sim_queries = [s["similar_query"] for s in sim_objs]
    base_vec = model.encode(base_query, convert_to_numpy=True)
    sim_queries_processed = []
    if sim_queries:
        sim_vecs = model.encode(sim_queries, convert_to_numpy=True, batch_size=32)
        sim_scores = cosine_similarity([base_vec], sim_vecs)[0]
        for sim_obj, sim_score in zip(sim_objs, sim_scores):
            record = {
                "similar_query": sim_obj["similar_query"],
                "jaccard_score": float(sim_obj["jaccard_score"]),
                "cosine_score": float(sim_score)
            }
            if threshold_cosine is None or sim_score >= threshold_cosine:
                sim_queries_processed.append(record)
        sim_queries_processed.sort(key=lambda x: x["cosine_score"], reverse=True)

    results_cosine.append({
        "base_query": base_query,
        "similar_queries": sim_queries_processed
    })

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results_cosine, f, indent=2, ensure_ascii=False)
print(f"Done. Saved {len(results_cosine)} entries to {output_file}")